In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
        

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [3]:
import pandas as pd

# Load the data using your correct paths!
train_data = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test_data = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

# Look at the first 5 rows
display(train_data.head())

# Check for missing values in our data
print("\n--- Missing values in the dataset ---")
print(train_data.isnull().sum())


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S



--- Missing values in the dataset ---
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [4]:
# 1. Fill missing 'Age' with the median age
train_data['Age'] = train_data['Age'].fillna(train_data['Age'].median())
test_data['Age'] = test_data['Age'].fillna(test_data['Age'].median())

# 2. Fill missing 'Embarked' with the most common value ('S')
train_data['Embarked'] = train_data['Embarked'].fillna('S')

# 3. Fill missing 'Fare' in the test set
test_data['Fare'] = test_data['Fare'].fillna(test_data['Fare'].median())

# 4. Save test PassengerIds (we need this for the final Kaggle submission)
test_passenger_ids = test_data['PassengerId']

# 5. Drop columns we aren't using right now
cols_to_drop = ['Cabin', 'Ticket', 'Name', 'PassengerId']
train_data = train_data.drop(cols_to_drop, axis=1)
test_data = test_data.drop(cols_to_drop, axis=1)

# Check our work! 
print("Missing values in Training Data:")
print(train_data.isnull().sum())

Missing values in Training Data:
Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64


In [5]:
# Convert text columns (Sex, Embarked) into numbers (1s and 0s)
train_data = pd.get_dummies(train_data, columns=['Sex', 'Embarked'])
test_data = pd.get_dummies(test_data, columns=['Sex', 'Embarked'])

# Let's see what the data looks like now!
display(train_data.head())

,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,0,3,22.0,1,0,7.2500,False,True,False,False,True
1,1,1,38.0,1,0,71.2833,True,False,True,False,False
2,1,3,26.0,0,0,7.9250,True,False,False,False,True
3,1,1,35.0,1,0,53.1000,True,False,False,False,True
4,0,3,35.0,0,0,8.0500,False,True,False,False,True


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Separate the features (X) from the target/answer (y)
y = train_data['Survived']
X = train_data.drop('Survived', axis=1)

# 2. Create the Random Forest model
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)

# 3. Train (fit) the model on our data!
model.fit(X, y)

# 4. Let's see how well it learned the training data
predictions = model.predict(X)
accuracy = accuracy_score(y, predictions)
print(f"Model Training Accuracy: {accuracy * 100:.2f}%")

Model Training Accuracy: 84.40%


In [7]:
# 1. Use the trained model to predict who survived in the test data
test_predictions = model.predict(test_data)

# 2. Create a new dataframe for the Kaggle submission
submission = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Survived': test_predictions
})

# 3. Save it to a CSV file
submission.to_csv('submission.csv', index=False)

print("Your submission was successfully saved!")
display(submission.head())

Your submission was successfully saved!


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 1. Feature Engineering: Create a 'FamilySize' column
train_data['FamilySize'] = train_data['SibSp'] + train_data['Parch'] + 1
test_data['FamilySize'] = test_data['SibSp'] + test_data['Parch'] + 1

# Re-separate the features and target after adding new columns
y = train_data['Survived']
X = train_data.drop('Survived', axis=1)

# 2. Split the data into Training (80%) and Validation (20%)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train a slightly deeper Random Forest
advanced_model = RandomForestClassifier(n_estimators=200, max_depth=7, random_state=42)
advanced_model.fit(X_train, y_train)

# 4. Predict on the Validation set
val_predictions = advanced_model.predict(X_val)

# 5. Print out ALL the metrics (Precision, Recall, F1, Accuracy)
print("--- Advanced Model Metrics ---")
print(classification_report(y_val, val_predictions))

--- Advanced Model Metrics ---
              precision    recall  f1-score   support

           0       0.80      0.90      0.85       105
           1       0.83      0.68      0.75        74

    accuracy                           0.81       179
   macro avg       0.82      0.79      0.80       179
weighted avg       0.81      0.81      0.81       179



In [9]:
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# 1. Reload the raw data so we have the 'Name' column back
train_data = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test_data = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

# Combine them temporarily so we can clean both at the same time
all_data = [train_data, test_data]

for dataset in all_data:
    # 2. Extract Titles from Names
    dataset['Title'] = dataset['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
    
    # Group rare titles together
    dataset['Title'] = dataset['Title'].replace(['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    dataset['Title'] = dataset['Title'].replace(['Mlle', 'Ms'], 'Miss')
    dataset['Title'] = dataset['Title'].replace('Mme', 'Mrs')
    
    # 3. Fill missing Age based on the median age for their specific Title!
    dataset['Age'] = dataset['Age'].fillna(dataset.groupby('Title')['Age'].transform('median'))
    
    # 4. Create an 'IsAlone' feature (single travelers had different survival rates)
    dataset['FamilySize'] = dataset['SibSp'] + dataset['Parch'] + 1
    dataset['IsAlone'] = 0
    dataset.loc[dataset['FamilySize'] == 1, 'IsAlone'] = 1
    
    # 5. Fill remaining blanks
    dataset['Embarked'] = dataset['Embarked'].fillna('S')
    dataset['Fare'] = dataset['Fare'].fillna(dataset['Fare'].median())

# 6. Drop the columns we don't need anymore
features_drop = ['Ticket', 'Cabin', 'Name', 'PassengerId']
train_data = train_data.drop(features_drop, axis=1)
test_data = test_data.drop(features_drop, axis=1)

# 7. One-Hot Encode (Text to Numbers)
train_data = pd.get_dummies(train_data, columns=['Sex', 'Embarked', 'Title'])

# 8. Train/Validation Split
X = train_data.drop('Survived', axis=1)
y = train_data['Survived']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 9. Train a Gradient Boosting Model (Kaggle favorite)
gbm_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gbm_model.fit(X_train, y_train)

# 10. Check Accuracy & Metrics
val_predictions = gbm_model.predict(X_val)
accuracy = accuracy_score(y_val, val_predictions)

print(f"Grandmaster Model Validation Accuracy: {accuracy * 100:.2f}%\n")
print("--- Detailed Metrics ---")
print(classification_report(y_val, val_predictions))

<>:15: SyntaxWarning: invalid escape sequence '\.'
<>:15: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_16/1297011607.py:15: SyntaxWarning: invalid escape sequence '\.'
  dataset['Title'] = dataset['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)


Grandmaster Model Validation Accuracy: 83.80%

--- Detailed Metrics ---
              precision    recall  f1-score   support

           0       0.85      0.88      0.86       105
           1       0.82      0.78      0.80        74

    accuracy                           0.84       179
   macro avg       0.83      0.83      0.83       179
weighted avg       0.84      0.84      0.84       179

